# Protein Prep 

This notebook shows you how to perform protein prep in six different scenarios


| # | Pockets | Loops | Platform `method` | Typical latency | This notebook |
|---|---------|-------|-------------------|-----------------|---------------|
| 1 | `no` | off | `direct` | fast | §1 — `1EBY`, loops off, no extract |
| 2 | `from-crystal-ligand` | off | `direct` | fast | §2 — `1EBY`, extract co-crystal ligand |
| 3 | `no` | on | `direct` | up to ~10 min | §3 — `5QSP`, loop modelling |
| 4 | `from-crystal-ligand` | on | `direct` | same as 3 | §4 — `1XKK`, loops on + extract |
| 5 | `novel` | off | `workflow` | slow (Argo + Pocket Finder) | §5 — `5QSP`, novel pockets |
| 6 | `novel` | on | `workflow` | same as 5 | §6 — `5QSP`, loops + novel pockets |

**Notes**

- Prepare requires a **registered** protein (`protein.sync()` before `run()` / `start()`).
- Structure assessment is **out of band** — use `StructureReport`, not `ProteinPrep`.
- Scenarios **5–6** must use `start()` / `watch()` (blocking `run()` is rejected for `find_pockets="novel"`).
- Optional: `start(quote=True)` then `confirm()` before `watch()` when exercising Pocket Finder billing on dev.



In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from deeporigin import config

config.set_env("dev")  # or "local" with mock server


In [ ]:
from deeporigin import projects
from deeporigin.drug_discovery import Protein, ProteinPrep, StructureReport
from deeporigin.platform import DeepOriginClient

projects.create("python-client-dev")
client = DeepOriginClient()
client


## Structure Report 

Protein Prep v10 does **not** emit `structure_reports[]`. Run `StructureReport` on the source structure when you need an assessment report before or after prepare.


In [ ]:
# Optional — quick source report on 5QSP before loop modelling scenarios
protein = Protein.from_pdb_id("5qsp")
protein.sync()
sr = StructureReport(protein=protein)
sr.run()[0]


## 1. No pockets, loops off 

In this example, we prepare a protein without loop modelling, and without finding pockets. 


In [ ]:
protein = Protein.from_pdb_id("1eby")
protein.sync()

prep = ProteinPrep(protein=protein, client=client)
prep.recommend()

In [ ]:
prep

In [ ]:
prep.keep(kind="ligand")  # retain ligand in receptor; no extract → no crystal pockets
prep.skip(decision="review")  # resolve remaining analyzer reviews (e.g. waters)
prepared = prep.run()

In [ ]:
prepared.download()
prepared.show()


## 2. Crystal ligand pockets, loops off 

In this example, we prepare a protein, extract a crystal ligand, find a pocket from the crystal ligand


In [ ]:
protein = Protein.from_pdb_id("1eby")
protein.sync()
prep = ProteinPrep(protein=protein, client=client)
prep.recommend()
(
    prep.keep(kind="water", subtype="coordinating")
    .skip(kind="water", subtype="crystal")
    .extract(kind="ligand")
)
prep

In [ ]:
prepared = prep.run()
prepared.download()
prepared.show()

In [ ]:
poses = prep.get_crystal_poses()
pockets = prep.get_pockets()

In [ ]:
poses[0].download()
poses[0].show() 

In [ ]:
prepared.show(poses=[poses[0]])

## 3. No pockets, loops on 

In this example, we're filling gaps using loop modelling. Here, we work with `5QSP`, and we see that there are gaps in the structure


In [ ]:
protein = Protein.from_pdb_id("5qsp")
protein.sync()
protein.show()


In [ ]:
prep = ProteinPrep(protein=protein, client=client)
prep.recommend()
prep.skip(decision="review")
prep.find_pockets="no"
prep


In [ ]:
prepared = prep.run()

In [ ]:
prepared.download()
prepared.show()


## 4. Crystal ligand + loops on 

Combine loop modelling with co-crystal pocket geometry.


In [ ]:
protein = Protein.from_pdb_id("1XKK")
protein.sync()
protein.show()

In [ ]:
prep = ProteinPrep(protein=protein, client=client)
prep.recommend()
(
    prep.keep(kind="water", subtype="coordinating")
    .skip(kind="water", subtype="crystal")
    .extract(kind="ligand")
)
prep

In [ ]:
prepared = prep.run()

In [ ]:
prepared.download()
prepared.show()

In [ ]:
pockets = prep.get_pockets()
prepared.show(pockets=pockets)

In [ ]:
poses = prep.get_crystal_poses()
poses.download()
prepared.show(poses=poses)

## 5. Novel pockets, loops off 

In this example, we find novel pockets using PocketFinder


In [ ]:
protein = Protein.from_pdb_id("3ERT")
protein.sync()
protein.show()


In [ ]:
prep = ProteinPrep(protein=protein, client=client)
prep.recommend()
prep.skip(decision="review")
prep

In [ ]:
prep.find_pockets = "novel"
prep


In [ ]:
prep.start()
await prep.watch()

In [ ]:
prepared = prep.get_results()
prepared.download()
prepared.show()


In [ ]:
pockets = prep.get_pockets()
prepared.show(pockets=pockets)

## 6. Novel pockets + loop modelling

Loop modelling runs in the served prepare child; Pocket Finder publishes `pockets[]` on the parent workflow execution.


In [ ]:
protein = Protein.from_pdb_id("5qsp")
protein.sync()
prep = ProteinPrep(protein=protein, client=client)
prep.recommend()
prep.skip(decision="review")
prep


In [ ]:
prep.find_pockets = "novel"
prep.pocket_count = 1
prep.pocket_min_size = 100
prep.start()


In [ ]:
await prep.watch()


In [ ]:
prepared = prep.get_results()
prepared.download()
prepared.show()


In [ ]:
pockets = prep.get_pockets()

In [ ]:
prepared.show(pockets=pockets)